# 8) Frequency and reserves

## Objectives

At the end of this notebook you should be able to:

- write the **swing equation** for a whole synchronous area and explain what **inertia** does in the first seconds after a plant trips
- simulate the frequency after an incident, with **frequency containment reserve** (FCR) and the load's own response, and check the simulation against two closed-form results: the initial **rate of change of frequency** (RoCoF) and the settled deviation
- test a system against Continental Europe's design criteria for its **reference incident** of 3,000 MW
- explain why falling inertia makes the same incident more dangerous, and how **frequency restoration reserve** (aFRR) brings the frequency back to 50 Hz

Every lesson so far has been a snapshot: a power flow, a dispatch, a security check at one moment. Frequency is about the seconds and minutes **after** something goes wrong — the one place in this module where time is the subject. The tool is a small dynamic simulation, written from the physics and checked against what the physics predicts.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

F0 = 50.0                                       # nominal frequency, Hz

### The swing equation

All synchronous machines in an interconnected area turn in step. Their rotating masses store kinetic energy, measured by the **inertia constant H**: the seconds for which that energy could supply the rated power S. When generation and load stop matching, the difference is drawn from — or pushed into — that energy, and the frequency moves:

**(2 H S / f₀) · dΔf/dt = ΔP_response − ΔP_incident − D · Δf**

- ΔP_incident: the lost generation (here, a plant trips);
- D · Δf: the load's own response — motors slow down and draw less, about 1 % of the load per Hz (**load self-regulation**);
- ΔP_response: the reserves. **FCR** responds to the frequency deviation itself, in proportion, until it is fully activated at 200 mHz, within seconds (here a first-order lag of 6 s); **aFRR** later integrates the deviation and brings the frequency back to 50 Hz.

Two results follow without any simulation. At the first instant nothing has responded yet, so **RoCoF = −ΔP · f₀ / (2 H S)**. Once FCR and load have settled, the frequency stops moving where their responses add up to the loss: **Δf_settled = −ΔP / (K_FCR + D)**, with K_FCR the FCR gain in GW per Hz.

In [2]:
def simulate(loss_gw, S_gw=300.0, H_s=5.0, fcr_gw=3.0, fcr_full_hz=0.2, T_fcr_s=6.0, self_regulation=0.01,
             Ki_afrr=0.0, fast_gw=0.0, T_fast_s=0.5, T_end=120.0, step=0.01):
    """One-area frequency model after a sudden loss of loss_gw (negative: a surplus).
    Returns a DataFrame: frequency deviation (Hz) and the responses of FCR, aFRR, a fast reserve and the load (GW)."""
    K_fcr = fcr_gw / fcr_full_hz                  # GW per Hz, until the reserve is exhausted
    D = self_regulation * S_gw                    # GW per Hz (the load is taken as S)
    K_fast = fast_gw / fcr_full_hz if fast_gw else 0.0

    def rhs(t, x):
        df, p_fcr, p_afrr, p_fast = x
        target_fcr = np.clip(-K_fcr * df, -fcr_gw, fcr_gw)
        target_fast = np.clip(-K_fast * df, -fast_gw, fast_gw)
        response = p_fcr + p_afrr + p_fast - D * df
        return [F0 / (2 * H_s * S_gw) * (response - loss_gw), (target_fcr - p_fcr) / T_fcr_s, -Ki_afrr * df,
                (target_fast - p_fast) / T_fast_s]

    t = np.linspace(0, T_end, int(round(T_end / step)) + 1)
    sol = solve_ivp(rhs, (0, T_end), [0.0, 0.0, 0.0, 0.0], t_eval=t, max_step=step, rtol=1e-9, atol=1e-11)
    df, p_fcr, p_afrr, p_fast = sol.y
    return pd.DataFrame({"df_Hz": df, "FCR_GW": p_fcr, "aFRR_GW": p_afrr, "fast_GW": p_fast,
                         "load_response_GW": -D * df}, index=pd.Index(sol.t, name="t_s"))


def summary(run):
    i = run.df_Hz.idxmin() if run.df_Hz.iloc[-1] < 0 else run.df_Hz.idxmax()
    rocof = (run.df_Hz.iloc[1] - run.df_Hz.iloc[0]) / (run.index[1] - run.index[0])
    return {"initial_RoCoF_Hz_per_s": rocof, "nadir_Hz": run.df_Hz[i], "nadir_time_s": i, "settled_Hz": run.df_Hz.iloc[-1]}

### The reference incident in Continental Europe

Continental Europe dimensions its FCR for a **reference incident** of 3,000 MW — the loss of the two largest units, roughly — and requires that the frequency then stays within **800 mHz** of 50 Hz at every instant and settles within **200 mHz**. The area's size and inertia below (300 GW, H = 5 s) are illustrative values of the right order; FCR is the area's 3,000 MW, fully activated at 200 mHz.

In [3]:
S, H, LOSS = 300.0, 5.0, 3.0
run = simulate(LOSS, S, H)
result = summary(run)
K_fcr, D = 3.0 / 0.2, 0.01 * S
analytic = {"initial_RoCoF_Hz_per_s": -LOSS * F0 / (2 * H * S), "settled_Hz": -LOSS / (K_fcr + D)}
assert abs(result["initial_RoCoF_Hz_per_s"] - analytic["initial_RoCoF_Hz_per_s"]) < 1e-3
assert abs(result["settled_Hz"] - analytic["settled_Hz"]) < 1e-4
print(f"initial RoCoF: simulated {result['initial_RoCoF_Hz_per_s']:.4f}, analytic {analytic['initial_RoCoF_Hz_per_s']:.4f} Hz/s")
print(f"settled:       simulated {result['settled_Hz']:.4f}, analytic {analytic['settled_Hz']:.4f} Hz")
print(f"nadir {result['nadir_Hz']:.3f} Hz after {result['nadir_time_s']:.1f} s  (limits: 0.8 Hz at any instant, 0.2 Hz settled)")
print(f"after 120 s: FCR {run.FCR_GW.iloc[-1]:.2f} GW, load response {run.load_response_GW.iloc[-1]:.2f} GW")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(run.index, F0 + run.df_Hz, color="tab:blue")
axes[0].axhline(F0 - 0.8, color="tab:red", ls="--", lw=1, label="instantaneous limit 49.2 Hz")
axes[0].axhline(F0 - 0.2, color="tab:orange", ls="--", lw=1, label="settled limit 49.8 Hz")
axes[0].set(xlabel="seconds after the incident", ylabel="frequency (Hz)", title="Loss of 3,000 MW", ylim=(49.1, 50.05))
axes[0].legend(fontsize=8)
axes[1].stackplot(run.index, run.FCR_GW, run.load_response_GW, labels=["FCR", "load self-regulation"],
                  colors=["tab:green", "tab:gray"], alpha=0.7)
axes[1].axhline(LOSS, color="black", ls="--", lw=1, label="lost generation")
axes[1].set(xlabel="seconds after the incident", ylabel="GW", title="Who replaces the lost 3 GW")
axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

initial RoCoF: simulated -0.0500, analytic -0.0500 Hz/s
settled:       simulated -0.1667, analytic -0.1667 Hz
nadir -0.247 Hz after 10.9 s  (limits: 0.8 Hz at any instant, 0.2 Hz settled)
after 120 s: FCR 2.50 GW, load response 0.50 GW


The simulation reproduces both closed-form results — an initial RoCoF of −0.0500 Hz/s and a settled deviation of −0.1667 Hz — which is the test that it implements the swing equation correctly. The frequency falls for 10.9 s to its **nadir**, 49.753 Hz, then recovers and settles at 49.833 Hz: within both limits, with room to spare. The right-hand plot shows how. In the first seconds almost nothing responds and the whole 3 GW comes out of the rotating masses' kinetic energy — that is what inertia is. FCR then ramps up; at the nadir, FCR and load together match the loss for the first time. Afterwards they briefly supply **more** than the loss — about 3.4 GW at 20 s — to refill the kinetic energy and lift the frequency back. In the end FCR carries 2.50 GW and the load 0.50 GW, by having dropped with the frequency.

The reference incident is a design case, and real incidents can exceed it. On 8 January 2021, the Continental Europe synchronous area split into two parts for about an hour; ENTSO-E's investigation found that the initial event was at the Ernestinovo substation in Croatia. The split left a deficit of about 6.3 GW in the North-West area and a surplus of the same size in the South-East: the North-West frequency fell to 49.74 Hz within about 15 seconds, the South-East's rose to as much as 50.6 Hz, and about 1.7 GW of contracted interruptible load in France and Italy was disconnected. The same imbalance moved the smaller South-East area much further — the size of a system matters, which the exercise's challenge takes up.

### Less inertia, same incident

Wind turbines, solar panels and batteries connect through power electronics; unless they are controlled to do so, they add no inertia. As they replace synchronous plants, H falls. The same 3,000 MW loss, with less and less inertia:

In [4]:
rows, runs = {}, {}
for h in [6.0, 5.0, 4.0, 3.0, 2.0]:
    runs[h] = simulate(LOSS, S, h, T_end=60.0)
    r = summary(runs[h])
    rows[h] = {"initial_RoCoF_Hz_per_s": r["initial_RoCoF_Hz_per_s"], "nadir_Hz": r["nadir_Hz"], "nadir_time_s": r["nadir_time_s"]}
inertia = pd.DataFrame(rows).T.rename_axis("H (s)")
fig, ax = plt.subplots(figsize=(7.5, 3.4))
for h, r in runs.items():
    ax.plot(r.index, F0 + r.df_Hz, label=f"H = {h:.0f} s")
ax.set(xlabel="seconds after the incident", ylabel="frequency (Hz)", xlim=(0, 40), title="The same incident with less inertia")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
inertia.round(3)

       initial_RoCoF_Hz_per_s  nadir_Hz  nadir_time_s
H (s)                                                
6.0                    -0.042    -0.231         11.95
5.0                    -0.050    -0.247         10.90
4.0                    -0.062    -0.271          9.78
3.0                    -0.083    -0.306          8.52
2.0                    -0.125    -0.363          6.99

With less inertia the same 3 GW is drawn from less stored energy, so the frequency falls faster: the initial RoCoF is inversely proportional to H, from −0.042 Hz/s at H = 6 s to −0.125 Hz/s at H = 2 s. The nadir deepens too, from −0.231 to −0.363 Hz, and comes earlier (12.0 s against 7.0 s) — but it does **not** scale with 1/H: halving H from 4 to 2 s doubles the RoCoF, yet deepens the nadir by only a third. FCR starts responding as soon as the frequency moves, and the faster the fall, the sooner the reserve's ramp catches up with it. Below some inertia, though, the fall outruns the reserve: the race between how fast the frequency falls and how fast the reserves respond is the core of the inertia debate. With falling inertia, operators can respond in three ways — keep synchronous machines online, add **synthetic inertia** or **fast frequency response** from converters (a battery can respond in a fraction of a second), or limit the largest possible loss. The exercise compares the first two.

### Back to 50 Hz: frequency restoration

FCR stops the fall, but leaves the frequency 0.17 Hz low and the FCR itself fully used — no reserve left for a second incident. Within minutes, **automatic frequency restoration reserve** (aFRR) takes over: each control area's controller integrates the remaining deviation and orders more generation until the frequency is back at 50 Hz and FCR is released. Continental Europe requires the frequency to be restored within 15 minutes.

In [5]:
restore = simulate(LOSS, S, H, Ki_afrr=0.15, T_end=900.0, step=0.05)      # aFRR gain: GW per Hz per second
for t in [30, 60, 120, 300, 600, 900]:
    row = restore.loc[restore.index[np.argmin(np.abs(restore.index - t))]]
    print(f"t = {t:4d} s: deviation {row.df_Hz * 1000:7.1f} mHz, FCR {row.FCR_GW:5.2f} GW, aFRR {row.aFRR_GW:5.2f} GW")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(restore.index / 60, F0 + restore.df_Hz, color="tab:blue")
axes[0].set(xlabel="minutes after the incident", ylabel="frequency (Hz)", title="FCR, then aFRR")
axes[1].stackplot(restore.index / 60, restore.FCR_GW, restore.aFRR_GW, restore.load_response_GW,
                  labels=["FCR", "aFRR", "load self-regulation"], colors=["tab:green", "tab:purple", "tab:gray"], alpha=0.7)
axes[1].set(xlabel="minutes after the incident", ylabel="GW", title="Hand-over from FCR to aFRR")
axes[1].legend(fontsize=8, loc="center right")
plt.tight_layout()
plt.show()

t =   30 s: deviation  -112.2 mHz, FCR  1.91 GW, aFRR  0.72 GW
t =   60 s: deviation   -97.0 mHz, FCR  1.54 GW, aFRR  1.22 GW
t =  120 s: deviation   -59.6 mHz, FCR  0.94 GW, aFRR  1.91 GW
t =  300 s: deviation   -13.6 mHz, FCR  0.21 GW, aFRR  2.75 GW
t =  600 s: deviation    -1.2 mHz, FCR  0.02 GW, aFRR  2.98 GW
t =  900 s: deviation    -0.1 mHz, FCR  0.00 GW, aFRR  3.00 GW


The hand-over takes minutes. After 30 s FCR still carries most of the response (1.91 GW, aFRR 0.72 GW); after 5 minutes aFRR carries 2.75 GW and the deviation is down to 13.6 mHz; after 10 minutes FCR is almost free again, and after 15 minutes the frequency is back at 50 Hz with aFRR carrying the whole 3 GW. FCR is a **shared, fast and short** reserve for the whole synchronous area; aFRR is slower, and in a real interconnected system each control area's aFRR replaces only its **own** imbalance — the controller acts on the area control error, which combines the frequency deviation with the area's deviation from its scheduled exchange, so that the area where the plant tripped pays it back while its neighbours' FCR is released. This one-area model merges the two, but the sequence is the same: inertia for the first seconds, FCR for the first minute, aFRR for the next quarter of an hour.

## Check your understanding

**Inertia and FCR**

1. In the first instant after a plant trips, neither FCR nor the load has responded. Where does the missing power come from?
2. Halving the inertia doubles the initial RoCoF. Why does it not double the nadir, too?

**Reserves**

1. Why does FCR alone leave the frequency below 50 Hz, and why is that a problem even if 49.83 Hz is within the limits?
2. On 8 January 2021 the imbalance after the split was about 6.3 GW, more than twice the reference incident. Why is a system separation a harder test than the loss of the largest plants?